## Manipulating 1 parquet file

In [56]:
import pyarrow.parquet as pq
import random
import os

In [3]:
parquet_file = pq.ParquetFile(r'data\data\train-00000-of-00006.parquet')
parquet_file.metadata

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 7
  num_rows: 18687
  num_row_groups: 5
  format_version: 2.6
  serialized_size: 5723

In [4]:
parquet_file.schema

required group field_id=-1 schema {
  optional binary field_id=-1 patient_id (String);
  optional int32 field_id=-1 scan_id;
  optional group field_id=-1 image {
    optional binary field_id=-1 bytes;
    optional binary field_id=-1 path (String);
  }
  optional int32 field_id=-1 age;
  optional binary field_id=-1 sex (String);
  optional group field_id=-1 labels (List) {
    repeated group field_id=-1 list {
      optional int32 field_id=-1 element;
    }
  }
}

In [73]:
path = r'data\data\train-00000-of-00006.parquet'
table = pq.read_table(path, columns=["patient_id", "labels"])
table

pyarrow.Table
patient_id: string
labels: list<element: int32>
  child 0, element: int32
----
patient_id: [["1","1","1","2","3",...,"1204","1204","1205","1205","1205"],["1205","1205","1205","1205","1205",...,"2345","2346","2347","2348","2348"],...,["3470","3471","3472","3473","3474",...,"4808","4808","4808","4808","4808"],["4808","4808","4808","4808","4808",...,"4959","4959","4960","4960","4960"]]
labels: [[[2],[2,11],...,[0],[14,4]],[[14],[0],...,[0],[0]],...,[[5],[0],...,[4],[0]],[[0],[0],...,[0],[0]]]

In [74]:
patient_ids = table["patient_id"].to_pylist()
labels_list = table["labels"].to_pylist()

In [79]:
#https://huggingface.co/datasets/chehablab/NIHChestXR/blob/main/README.md
#Pneumonia is 7
random.seed(42)
patient_ids_pneumonia = [patient_id for patient_id, labels in zip(patient_ids, labels_list) if 7 in labels]
patient_ids_pneumonia = sorted(set(patient_ids_pneumonia))
print(len(patient_ids_pneumonia))
unique_ids = sorted(set(patient_ids))
print(len(unique_ids))
patient_ids_nopneumonia = list(set(unique_ids) - set(patient_ids_pneumonia))
print(len(patient_ids_nopneumonia))
random.shuffle(patient_ids_nopneumonia)
random.shuffle(patient_ids_pneumonia)
patient_ids_train = patient_ids_nopneumonia[:int(len(patient_ids_nopneumonia)*0.8)] + patient_ids_pneumonia[:int(len(patient_ids_pneumonia)*0.8)]
patient_ids_val = patient_ids_nopneumonia[int(len(patient_ids_nopneumonia)*0.8):int(len(patient_ids_nopneumonia)*0.9)] + patient_ids_pneumonia[int(len(patient_ids_pneumonia)*0.8):int(len(patient_ids_pneumonia)*0.9)]
patient_ids_test = patient_ids_nopneumonia[int(len(patient_ids_nopneumonia)*0.9):] + patient_ids_pneumonia[int(len(patient_ids_pneumonia)*0.9):]
print(len(train), len(val), len(test))

162
4960
4798
3967 496 497


In [80]:
print("Train-validation overlap:", len(set(patient_ids_train) & set(patient_ids_val)))
print("Train-test overlap:", len(set(patient_ids_train) & set(patient_ids_test)))
print("Validation-test overlap:", len(set(patient_ids_val) & set(patient_ids_test)))

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [89]:
patient_ids_train = set(patient_ids_train)
patient_ids_val = set(patient_ids_val)
patient_ids_test = set(patient_ids_test)
train_data = []
val_data=[]
test_data = []
for i, (patient_id, labels) in enumerate(
        zip(patient_ids, labels_list)
    ):
    if patient_id is None:
        raise ValueError(f"Missing patient ID in {file.name}, row {i}")
    pneumonia = int(7 in labels)
    row = {
            "row_index": i,
            "patient_id": patient_id,
            "pneumonia": pneumonia,
        }
    if patient_id in patient_ids_train:
        train_data.append(row)
    elif patient_id in patient_ids_val:
        val_data.append(row)
    elif patient_id in patient_ids_test:
        test_data.append(row)
    else:
        raise ValueError(f"Patient {patient_id} is not assigned to a split")

## Manipulating all six files

Your split assigns patients to sets, but it doesn't yet assign their images to those sets. In the next step, we'll create image-level metadata across all six shards, including each image's patient ID, pneumonia label, shard filename, and row index. Then we can use the patient lists to determine which split each image belongs to.

In [97]:
data_dir = r'data\data'
parquet_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".parquet")]
random.seed(42)

patient_ids_pneumonia = []
unique_ids = []
for file in parquet_files:
    table = pq.read_table(
        file,
        columns=["patient_id", "labels"]
    )
    patient_ids = table["patient_id"].to_pylist()
    labels_list = table["labels"].to_pylist()
    patient_ids_pneumonia = patient_ids_pneumonia + list(set([patient_id for patient_id, labels in zip(patient_ids, labels_list) if 7 in labels]))
    unique_ids = unique_ids + list(set(patient_ids))
patient_ids_pneumonia = sorted(set(patient_ids_pneumonia))
unique_ids = sorted(set(unique_ids))
print(len(patient_ids_pneumonia))
print(len(unique_ids))
patient_ids_nopneumonia = sorted(set(unique_ids) - set(patient_ids_pneumonia))
print(len(patient_ids_nopneumonia))
random.shuffle(patient_ids_nopneumonia)
random.shuffle(patient_ids_pneumonia)
patient_ids_train = patient_ids_nopneumonia[:int(len(patient_ids_nopneumonia)*0.8)] + patient_ids_pneumonia[:int(len(patient_ids_pneumonia)*0.8)]
patient_ids_val = patient_ids_nopneumonia[int(len(patient_ids_nopneumonia)*0.8):int(len(patient_ids_nopneumonia)*0.9)] + patient_ids_pneumonia[int(len(patient_ids_pneumonia)*0.8):int(len(patient_ids_pneumonia)*0.9)]
patient_ids_test = patient_ids_nopneumonia[int(len(patient_ids_nopneumonia)*0.9):] + patient_ids_pneumonia[int(len(patient_ids_pneumonia)*0.9):]
print(len(patient_ids_train), len(patient_ids_val), len(patient_ids_test))

print("Train-validation overlap:", len(set(patient_ids_train) & set(patient_ids_val)))
print("Train-test overlap:", len(set(patient_ids_train) & set(patient_ids_test)))
print("Validation-test overlap:", len(set(patient_ids_val) & set(patient_ids_test)))

patient_ids_train = set(patient_ids_train)
patient_ids_val = set(patient_ids_val)
patient_ids_test = set(patient_ids_test)
train_data = []
val_data=[]
test_data = []
for file in parquet_files:
    table = pq.read_table(
        file,
        columns=["patient_id", "labels"]
    )
    patient_ids = table["patient_id"].to_pylist()
    labels_list = table["labels"].to_pylist()
    for i, (patient_id, labels) in enumerate(
            zip(patient_ids, labels_list)
        ):
        if patient_id is None:
            raise ValueError(f"Missing patient ID in {os.path.basename(file)}, row {i}")
        pneumonia = int(7 in labels)
        row = {
                "shard": os.path.basename(file),
                "row_index": i,
                "patient_id": patient_id,
                "pneumonia": pneumonia,
            }
        if patient_id in patient_ids_train:
            train_data.append(row)
        elif patient_id in patient_ids_val:
            val_data.append(row)
        elif patient_id in patient_ids_test:
            test_data.append(row)
        else:
            raise ValueError(f"Patient {patient_id} is not assigned to a split")

1008
30805
29797
24643 3081 3081
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [98]:
print("Train images:", len(train_data))
print("Validation images:", len(val_data))
print("Test images:", len(test_data))
print("Total images:", len(train_data) + len(val_data) + len(test_data))

Train images: 90256
Validation images: 10923
Test images: 10941
Total images: 112120


In [99]:
train_patients = {row["patient_id"] for row in train_data}
val_patients = {row["patient_id"] for row in val_data}
test_patients = {row["patient_id"] for row in test_data}

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print("Patient-level leakage check passed.")

Patient-level leakage check passed.


In [100]:
import pandas as pd
os.makedirs("metadata", exist_ok=True)

train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
test_df = pd.DataFrame(test_data)

train_df.to_csv("metadata/train.csv", index=False)
val_df.to_csv("metadata/val.csv", index=False)
test_df.to_csv("metadata/test.csv", index=False)